# 27 – SQLAlchemy 2.0 Modern ORM

## Learning objectives

- Model tables with `DeclarativeBase`, `Mapped`, and `mapped_column`.
- Compare sync `Session` workflows with async `AsyncSession` patterns.
- Run the runnable demos under `examples/library/sqlalchemy_models/`.

## About this topic

SQLAlchemy 2.0 encourages typed columns via `Mapped[T]` and `mapped_column()`, which play well with static checkers. Prefer `select()` plus `session.scalars()` over legacy `query()`. Async engines pair with ASGI stacks; lifecycle rules (short-lived sessions, explicit commits) stay the same.

In [ ]:
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/sqlalchemy_models/declarative_base.py"
spec = importlib.util.spec_from_file_location("decl", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)
mod.main()

In [ ]:
# Typed ORM sketch aligned with the example module (in-memory SQLite)
from sqlalchemy import ForeignKey, String, create_engine, select
from sqlalchemy.orm import DeclarativeBase, Mapped, mapped_column, relationship, sessionmaker

class Base(DeclarativeBase):
    pass

class Team(Base):
    __tablename__ = "teams"
    id: Mapped[int] = mapped_column(primary_key=True)
    name: Mapped[str] = mapped_column(String(80), unique=True)
    members: Mapped[list["Member"]] = relationship(back_populates="team")

class Member(Base):
    __tablename__ = "members"
    id: Mapped[int] = mapped_column(primary_key=True)
    handle: Mapped[str] = mapped_column(String(80))
    team_id: Mapped[int] = mapped_column(ForeignKey("teams.id"))
    team: Mapped[Team] = relationship(back_populates="members")

engine = create_engine("sqlite:///:memory:", echo=False)
Base.metadata.create_all(engine)
Session = sessionmaker(engine, expire_on_commit=False)
with Session() as s:
    t = Team(name="agents")
    s.add_all([t, Member(handle="alpha", team=t), Member(handle="beta", team=t)])
    s.commit()
with Session() as s:
    team = s.scalars(select(Team).where(Team.name == "agents")).one()
    print(team.name, [m.handle for m in team.members])

In [ ]:
import asyncio
from pathlib import Path
import importlib.util

def _repo_root() -> Path:
    cwd = Path.cwd().resolve()
    return cwd if (cwd / "pyproject.toml").exists() else cwd.parent

path = _repo_root() / "examples/library/sqlalchemy_models/async_session.py"
spec = importlib.util.spec_from_file_location("async_sess", path)
mod = importlib.util.module_from_spec(spec)
spec.loader.exec_module(mod)

asyncio.run(mod.demo())

## Exercises and next steps

1. Add a composite index on `(team_id, handle)` on `Member` using `__table_args__`.
2. Swap the async DSN to `postgresql+asyncpg://...` in a copy of the async example and note what driver packages you need.
3. Read `examples/library/sqlalchemy_models/mixins_and_base_classes.py` and list which mixins you would reuse for audit columns.